# APTOS EfficientNet-B0 training (Google Colab GPU)

This notebook runs the repository's existing `training.train` pipeline on a Colab GPU. It uses the existing `data/splits/train.csv` and `val.csv` files and the shared `app_core.preprocess_array` preprocessing. It selects `models/best.pt` by validation quadratic weighted kappa (QWK). **It does not copy, open, or evaluate `test.csv` or test-split images.**

## Before running
1. In Colab, choose **Runtime → Change runtime type → T4 GPU** (or another available GPU).
2. Upload the project folder to Google Drive as `MyDrive/DR_screening_project/`. It must contain `app_core.py`, `training/train.py`, `training/__init__.py`, `data/splits/train.csv`, `data/splits/val.csv`, and `data/train_images/`. Keep the original `data/splits/test.csv` and test images in Drive; the staging cell below deliberately excludes them.
3. If your Drive folder has a different name or location, edit `PROJECT_DRIVE` in the next cell. The notebook copies only the code, train/validation CSVs, and images referenced by those two CSVs into Colab's temporary runtime.
4. After training, the notebook saves the checkpoint and log back to Drive and prompts a download of `best.pt`. Copy that file into your local project at `D:\\2_Projects\\DR_screening_project\\models\\best.pt` (create `models` if needed).
5. Only after training and checkpoint selection are complete, run the one-time held-out evaluation locally: ` .\\.venv\\Scripts\\python.exe -m scripts.evaluate`. Do not use test results to choose epochs, checkpoints, or settings.

Training may take several minutes or longer. Do not interrupt the cell before its completion message and checkpoint-save cell. A disconnected Colab runtime may lose its temporary files; the Drive copy is the persistent backup.

In [ ]:
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')
PROJECT_DRIVE = Path('/content/drive/MyDrive/DR_screening_project')  # Change if your Drive folder differs
if not PROJECT_DRIVE.is_dir():
    raise FileNotFoundError(f'Project folder not found: {PROJECT_DRIVE}. Upload the project folder to Drive or edit PROJECT_DRIVE.')
print('Drive project folder:', PROJECT_DRIVE)

In [ ]:
import importlib.util
import subprocess
import sys

# Colab normally provides PyTorch, torchvision, NumPy, and Pillow. Install only missing runtime packages.
required = {'torch': 'torch', 'torchvision': 'torchvision', 'numpy': 'numpy', 'PIL': 'Pillow',
            'pandas': 'pandas', 'cv2': 'opencv-python-headless', 'sklearn': 'scikit-learn'}
missing = [distribution for module, distribution in required.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', *missing])

import torch
if not torch.cuda.is_available():
    raise RuntimeError('No CUDA GPU is available. In Colab select Runtime → Change runtime type → GPU, then rerun this cell.')
print('GPU:', torch.cuda.get_device_name(0))
print('PyTorch:', torch.__version__, '| torchvision:', __import__('torchvision').__version__)

In [ ]:
import shutil
import pandas as pd

train_csv = PROJECT_DRIVE / 'data' / 'splits' / 'train.csv'
val_csv = PROJECT_DRIVE / 'data' / 'splits' / 'val.csv'
drive_images = PROJECT_DRIVE / 'data' / 'train_images'
for required_path in [PROJECT_DRIVE / 'app_core.py', PROJECT_DRIVE / 'training' / 'train.py',
                      PROJECT_DRIVE / 'training' / '__init__.py', train_csv, val_csv, drive_images]:
    if not required_path.exists():
        raise FileNotFoundError(f'Required project file/folder missing: {required_path}')

# Read only train and validation manifests. The held-out test manifest is intentionally not opened.
train_df = pd.read_csv(train_csv)
val_df = pd.read_csv(val_csv)
for name, frame in [('train', train_df), ('validation', val_df)]:
    if not {'id_code', 'diagnosis'}.issubset(frame.columns):
        raise ValueError(f'{name}.csv must contain id_code and diagnosis columns')
    frame['diagnosis'] = pd.to_numeric(frame['diagnosis'], errors='raise').astype(int)
    if not frame['diagnosis'].between(0, 4).all():
        raise ValueError(f'{name}.csv has a label outside 0..4')
if set(train_df.id_code) & set(val_df.id_code):
    raise ValueError('Train/validation ID overlap found; refusing to train.')

WORK_DIR = Path('/content/dr_screening_project_training')
if WORK_DIR.exists():
    raise FileExistsError('Staging directory already exists. Restart the Colab runtime before rerunning staging to avoid mixing artifacts.')
(WORK_DIR / 'training').mkdir(parents=True)
(WORK_DIR / 'data' / 'splits').mkdir(parents=True)
(WORK_DIR / 'data' / 'train_images').mkdir(parents=True)
shutil.copy2(PROJECT_DRIVE / 'app_core.py', WORK_DIR / 'app_core.py')
shutil.copy2(PROJECT_DRIVE / 'training' / 'train.py', WORK_DIR / 'training' / 'train.py')
shutil.copy2(PROJECT_DRIVE / 'training' / '__init__.py', WORK_DIR / 'training' / '__init__.py')
train_df.to_csv(WORK_DIR / 'data' / 'splits' / 'train.csv', index=False)
val_df.to_csv(WORK_DIR / 'data' / 'splits' / 'val.csv', index=False)

# Stage only train/validation images. No test CSV or test-only image is copied into the training runtime.
ids = list(dict.fromkeys([*train_df.id_code.tolist(), *val_df.id_code.tolist()]))
missing_images = [f'{image_id}.png' for image_id in ids if not (drive_images / f'{image_id}.png').is_file()]
if missing_images:
    raise FileNotFoundError(f'{len(missing_images)} train/validation images are missing; first missing: {missing_images[0]}')
for image_id in ids:
    shutil.copy2(drive_images / f'{image_id}.png', WORK_DIR / 'data' / 'train_images' / f'{image_id}.png')

print(f'Staged {len(train_df)} train and {len(val_df)} validation rows ({len(ids)} images).')
print('Train class counts:', train_df.diagnosis.value_counts().sort_index().to_dict())
print('Validation class counts:', val_df.diagnosis.value_counts().sort_index().to_dict())
print('Held-out test manifest and test-only images were excluded from the staging copy.')

## Train

This invokes the repository's existing script in the staged project. It uses the same deterministic preprocessing and ImageNet normalization as inference, augmentation only on training samples, a pretrained EfficientNet-B0 five-class head, fixed seed 42, and validation QWK checkpoint selection. The test split is not present in this runtime. The script first benchmarks five GPU batches and then runs 10 epochs.

In [ ]:
import os
os.chdir(WORK_DIR)
!python -u -m training.train --epochs 10 --batch-size 8 --seed 42

In [ ]:
import json
import shutil

CHECKPOINT_LOCAL = WORK_DIR / 'models' / 'best.pt'
LOG_LOCAL = WORK_DIR / 'models' / 'training_log.jsonl'
if not CHECKPOINT_LOCAL.is_file() or not LOG_LOCAL.is_file():
    raise FileNotFoundError('Training did not produce both best.pt and training_log.jsonl. Do not report training as complete.')
with LOG_LOCAL.open(encoding='utf-8') as log_file:
    epoch_rows = [json.loads(line) for line in log_file if line.strip()]
if len(epoch_rows) != 10 or epoch_rows[-1].get('epoch') != 10:
    raise RuntimeError(f'Expected all 10 epochs to finish; found {len(epoch_rows)} log rows. Do not report training as complete.')

# Copy completed artifacts back to Drive so they survive the temporary Colab runtime.
DRIVE_MODELS = PROJECT_DRIVE / 'models'
DRIVE_MODELS.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DRIVE = DRIVE_MODELS / 'best.pt'
shutil.copy2(CHECKPOINT_LOCAL, CHECKPOINT_DRIVE)
shutil.copy2(LOG_LOCAL, DRIVE_MODELS / 'training_log.jsonl')
saved = torch.load(CHECKPOINT_LOCAL, map_location='cpu', weights_only=False)
print('All 10 training epochs finished; best checkpoint saved to:', CHECKPOINT_DRIVE)
print('Selected epoch:', saved.get('epoch'), '| validation QWK:', saved.get('val_qwk'))
print('These are training/validation artifacts only; no held-out test metric was computed.')

## Download `best.pt` to your local project

Run the next cell after the artifact-copy cell succeeds. Colab will download the checkpoint through your browser. Save or move it to `D:\\2_Projects\\DR_screening_project\\models\\best.pt` on your computer. The same checkpoint and `training_log.jsonl` are also in the project's Google Drive `models/` folder. Then, from your local project root, run ` .\\.venv\\Scripts\\python.exe -m scripts.evaluate` exactly once for the held-out test evaluation. The evaluation script writes `models/test_metrics.json` and refuses to overwrite an existing report. Keep all test metrics out of model/epoch selection.

In [ ]:
from google.colab import files
files.download(str(CHECKPOINT_LOCAL))